In [1]:
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime
from googletrans import Translator


In [2]:
async def translate(sentense, from_lang='pt', to_lang='en'):
    translator = Translator()
    text_to_translate = await translator.translate(sentense, src=from_lang, dest=to_lang)
    return text_to_translate.text

In [3]:
url='https://jtm.com.mo/'
headers={'USER-AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36 Edg/128.0.0.0'}
r=requests.get(url, headers=headers)
r

<Response [200]>

In [4]:
soup=BeautifulSoup(r.content)
posts=soup.find_all('div', {'class': re.compile('title|featured-posts')})
valid_post=[]
for post in posts:
    if post.find('a'):
        valid_post.append(post.find('a', href=True).get('href'))

In [5]:
def get_article(url):
    headers={'USER-AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36 Edg/128.0.0.0'}
    r=requests.get(url, headers=headers)
    soup=BeautifulSoup(r.content)
    data=soup.find('div', {'id': 'main_body'})
    if data.find('span', {'class': 'meta-date'}):
        date=data.find('span', {'class': 'meta-date'}).text
        date=datetime.strptime(date, '%d %b, %Y').strftime('%Y-%m-%d %H:%M:%S')
    else:
        date=None
    
    if data.find('h2', {'class': 'single-title'}):
        title=data.find('h2', {'class': 'single-title'}).text
    else:
        title=None

    if data.find('span', {'class': 'meta-author'}):
        author=data.find('span', {'class': 'meta-author'}).text
    else:
        author=None

    if data.find('div', {'class': 'post-content entry clearfix'}):
        content=data.find('div', {'class': 'post-content entry clearfix'}).text
    else:
        content=None
    return date, title, author, content

In [6]:
# para=post[-1].split('\n')
# '\n'.join([await translate(p) for p in para])

In [7]:
restructured_posts=[]
for idx, post in enumerate(valid_post):
    date, title, author, content=get_article(post)
    link=post
    para=content.strip().split('\n')
    translated_content='\n'.join([await translate(p) for p in para])
    dict_post={
        # 'title': title,
        'title': await translate(title),
        'author': author,
        'link': link,
        'date': date,
        # 'content': content
        # 'content': await translate(content)
        'content': translated_content
    }
    restructured_posts.append(dict_post)

In [8]:

import xml.etree.ElementTree as ET

# Create the root of the RSS feed
rss = ET.Element('rss', version='2.0')
channel = ET.SubElement(rss, 'channel')

# Add channel elements
ET.SubElement(channel, 'title').text = 'Jornal TRIBUNA DE MACAU EN'
ET.SubElement(channel, 'link').text = url
ET.SubElement(channel, 'description').text = 'Jornal TRIBUNA DE MACAU EN RSS'

# Add items
for item in restructured_posts:
    item_elem = ET.SubElement(channel, 'item')
    ET.SubElement(item_elem, 'title').text = item['title']
    ET.SubElement(item_elem, 'link').text = item['link']
    ET.SubElement(item_elem, 'description').text = item['content']
    ET.SubElement(item_elem, 'pubDate').text = item['date']

# Convert to string and save to an XML file
rss_feed = ET.tostring(rss, encoding='utf-8', method='xml').decode()
with open('Jornal_TRIBUNA_DE_MACAU_en.xml', 'w', encoding='utf-8') as xml_file:
    xml_file.write(rss_feed)

In [9]:
import base64

# Variables
token = 'Replaced by real toekn'
repo = 'mogcai/CPC_RSS'  # Replace with your GitHub username/repo
commit_message = 'Update RSS feed'
file_path='Jornal_TRIBUNA_DE_MACAU_en.xml'

# Read the file content
with open(file_path, 'rb') as f:
    content = f.read()


In [10]:

# Encode the content
encoded_content = base64.b64encode(content).decode()
github_path='Jornal_TRIBUNA_DE_MACAU_en.xml'
# Prepare the API endpoint URL
url = f'https://api.github.com/repos/{repo}/contents/{github_path}'



In [11]:

# Fetch the current SHA of the file (if it exists)
response = requests.get(url, headers=headers)
if response.status_code == 200:
    file_info = response.json()
    sha = file_info['sha']  # Get the SHA of the file
else:
    sha = None  # File does not exist
    print(f"Error fetching file info: {response.content.decode()}")


In [12]:

# Prepare the payload
payload = {
    'message': commit_message,
    'content': encoded_content,
    'branch': 'main',  # Change to 'master' if that is your default branch
}

if sha:
    payload['sha'] = sha  # Include the SHA if the file exists



# Make the API request to create or update the file
headers = {
    'Authorization': f'token {token}',
    'Accept': 'application/vnd.github.v3+json'
}


In [13]:
import json
response = requests.put(url, headers=headers, data=json.dumps(payload))

# Check the response
if response.status_code == 201:
    print('File uploaded successfully.')
elif response.status_code == 200:
    print('File updated successfully.')
else:
    print(f'Error: {response.content}')


File updated successfully.
